In this notebook, we compute the results of ablating each base predictor class from the ensemble. We compute the bootstrapped RMSE on 10,000 samples with its 95% confidence interval on the holdout set.

In [1]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import rcParams
import numpy as np
import pandas as pd
import os

OUTPUT_FOLDER = "figures"

In [2]:
def compute_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

def bootstrapped_rmse(y_true, y_pred, n_iterations=10000, ci=95):
    boot_stats = []
    n_size = len(y_true)
    
    for _ in range(n_iterations):
        indices = np.random.choice(n_size, size=n_size, replace=True)
        resampled_true = y_true[indices]
        resampled_pred = y_pred[indices]
        
        score = compute_rmse(resampled_true, resampled_pred)
        boot_stats.append(score)
    
    # Calculate confidence intervals
    lower_bound = np.percentile(boot_stats, (100 - ci) / 2)
    upper_bound = np.percentile(boot_stats, 100 - (100 - ci) / 2)
    
    return np.mean(boot_stats), (lower_bound, upper_bound)

In [3]:
files = {
    "full": "/home/jusdai/cil-project/results/benchmark/full_pipeline.npz",
    "no_svdpp": "/home/jusdai/cil-project/results/ablations/components_no_svdpp.npz",
    "no_bagged_svdpp": "/home/jusdai/cil-project/results/ablations/components_no_bagged_svdpp.npz",
    "no_ease": "/home/jusdai/cil-project/results/ablations/components_no_ease.npz",
    "no_knn": "/home/jusdai/cil-project/results/ablations/components_no_knn.npz",
    "no_cold_start": "/home/jusdai/cil-project/results/ablations/components_no_cold_start.npz"
}

In [4]:
results_list = []

for name, path in files.items():
    data = np.load(path)
    preds = data["holdout_pred"]
    targets = data["y_holdout"]
    
    mean_rmse, conf_interval = bootstrapped_rmse(targets, preds)
    
    results_list.append({
        "Model": name,
        "RMSE": mean_rmse,
        "Lower": conf_interval[0],
        "Upper": conf_interval[1],
        "Error": (conf_interval[1] - conf_interval[0]) / 2 
    })

df_results = pd.DataFrame(results_list)

In [5]:
df_results

,Model,RMSE,Lower,Upper,Error
0,full,0.874415,0.871647,0.877250,0.002801
1,no_svdpp,0.880783,0.878004,0.883595,0.002796
2,no_bagged_svdpp,0.874676,0.871930,0.877485,0.002778
3,no_ease,0.875669,0.872939,0.878394,0.002727
4,no_knn,0.877929,0.875105,0.880739,0.002817
5,no_cold_start,0.874030,0.871268,0.876806,0.002769
